In [36]:

# ЧАСТЬ 1: ЗАПИСЬ И ПРЕОБРАЗОВАНИЕ ДАННЫХ

!pip install pandasql
import pandas as pd
from pandasql import sqldf
import numpy as np
import matplotlib.pyplot as plt



In [ ]:
# ------------------------------------------------------------
# 1. Настройки
# ------------------------------------------------------------
PATH = 'C:/Users/umed/Desktop/Работа по профессии/sandbox.xlsx'
SHEET = 'Лист1'   # ← ИМЯ ЛИСТА

# ------------------------------------------------------------
# 2. Читаем заголовки (4-я строка)
# ------------------------------------------------------------
df_headers = pd.read_excel(PATH, sheet_name=SHEET, skiprows=3, header=0, nrows=0)
columns = df_headers.columns.tolist()
print("Заголовки:", columns)

# ------------------------------------------------------------
# 3. Читаем df1 (строки 5-32)
# ------------------------------------------------------------
df1 = pd.read_excel(
    PATH, sheet_name=SHEET,
    skiprows=4, header=None, nrows=28, names=columns,
)

# ------------------------------------------------------------
# 4. Читаем df2 (строки 33-60)
# ------------------------------------------------------------
df2 = pd.read_excel(
    PATH, sheet_name=SHEET,
    skiprows=32, header=None, nrows=28, names=columns,
)

# ------------------------------------------------------------
# 5. Словарь месяцев
# ------------------------------------------------------------
months = {
    'янв': '01', 'фев': '02', 'мар': '03', 'апр': '04',
    'май': '05', 'июн': '06', 'июл': '07', 'авг': '08',
    'сен': '09', 'окт': '10', 'ноя': '11', 'дек': '12',
}

# ------------------------------------------------------------
# 6. Функция преобразования дат
# ------------------------------------------------------------
def convert_date(date_value, year=2026):
    if isinstance(date_value, pd.Timestamp):
        return date_value
    if isinstance(date_value, (int, float)):
        return pd.NaT
    date_str = str(date_value).strip()
    if '.' in date_str and any(m in date_str for m in months.keys()):
        parts = date_str.split('.')
        if len(parts) == 2:
            day, month_str = parts
            month = months.get(month_str.lower())
            if month:
                return f"{year}-{month}-{day.zfill(2)}"
    if '-' in date_str:
        return date_str
    return pd.NaT

# ------------------------------------------------------------
# 7. Преобразуем даты в df1 и df2
# ------------------------------------------------------------
for df in [df1, df2]:
    first_col = df.columns[0]
    df[first_col] = df[first_col].apply(convert_date)
    df[first_col] = pd.to_datetime(df[first_col], errors='coerce')
    df.rename(columns={first_col: 'date'}, inplace=True)
    df.dropna(how='all', inplace=True)

# ============================================================
# ЧАСТЬ 2: ПОДСЧЁТ ПОКАЗАТЕЛЕЙы
# ============================================================

# ------------------------------------------------------------
# 8. Пары колонок
# ------------------------------------------------------------
pairs = [
    ('fpd_01_shis', 'cum_znam_01', 'delta_01'),
    ('fpd_05_shis', 'cum_znam_05', 'delta_05'),ы
    ('fpd_07_shis', 'cum_znam_07', 'delta_07'),
]

# ------------------------------------------------------------
# 9. Кумулятивный подсчёт и дельта
# ------------------------------------------------------------
for df in [df1, df2]:
    for fpd_col, debt_col, delta_col in pairs:
        fpd_cumsum = df[fpd_col].cumsum()
        debt_cumsum = df[debt_col].cumsum()
        df[delta_col] = fpd_cumsum / debt_cumsum

# ------------------------------------------------------------
# 10. Сравнение befor vs after
# ------------------------------------------------------------
df1['period'] = 'befor'
df2['period'] = 'after'
df_all = pd.concat([df1, df2], ignore_index=True)

summary = df_all.groupby('period')[['delta_01', 'delta_05', 'delta_07']].mean()
print("Средние дельты (befor vs after):")
print(summary)

# ------------------------------------------------------------
# 11. График
# ------------------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(14, 15))
deltas = ['delta_01', 'delta_05', 'delta_07']
titles = ['Группа 1: FPD_01', 'Группа 2: FPD_05', 'Группа 3: FPD_07']

for ax, delta, title in zip(axes, deltas, titles):
    ax.plot(df1['date'], df1[delta], label='befor', color='blue')
    ax.plot(df2['date'], df2[delta], label='after', color='orange')
    ax.set_title(title)
    ax.set_xlabel('Дата')
    ax.set_ylabel('Delta')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()ы

# Версия 2

# Считывание DataFrame

In [33]:
# Считываем данные

# Глобальные переменные
PATH = 'C:/Users/umed/Desktop/Работа по профессии/sandbox.xlsx' 
cols = ['agreement_date', 'Sub_segment','znam_fpd0','chis_fpd0','znam_fpd5','chis_fpd5','znam_fpd7','chis_fpd7']
sheet_name = 'base'

df = pd.read_excel(PATH, 
             usecols=cols,
             sheet_name=sheet_name,)
df

,Sub_segment,agreement_date,znam_fpd0,chis_fpd0,znam_fpd5,chis_fpd5,znam_fpd7,chis_fpd7
0,Online_refferal,2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN
1,Online_Simplified,2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN
2,Offline_Direct,2026-08-30,NaN,NaN,NaN,NaN,NaN,NaN
3,Offline_standart,2026-08-30,NaN,NaN,NaN,NaN,NaN,NaN
4,Online_refferal,2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN
5,Online_Simplified,2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN
6,Offline_Direct,2026-08-30,NaN,NaN,NaN,NaN,NaN,NaN
7,Offline_standart,2026-08-30,NaN,NaN,NaN,NaN,NaN,NaN
8,Online_refferal,2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN
9,Online_Simplified,2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN


# Данные фильтруем



## Online
Отбираем только Online сегмент

In [41]:
df_on = sqldf("""SELECT * FROM df WHERE Sub_segment LIKE '%Online_%' GROUP BY agreement_date""")
df_on

,Sub_segment,agreement_date,znam_fpd0,chis_fpd0,znam_fpd5,chis_fpd5,znam_fpd7,chis_fpd7
0,Online_refferal,2026-01-01 00:00:00.000000,None,None,None,None,None,None
1,Online_refferal,2026-08-30 00:00:00.000000,None,None,None,None,None,None
2,Online_refferal,2026-08-31 00:00:00.000000,None,None,None,None,None,None
3,Online_Simplified,2026-09-01 00:00:00.000000,None,None,None,None,None,None
4,Online_refferal,2026-09-04 00:00:00.000000,None,None,None,None,None,None
5,Online_Simplified,2026-09-05 00:00:00.000000,None,None,None,None,None,None


## Offline
Отбираем только Оффлайн сегмент

In [42]:
df_of = sqldf("""SELECT * FROM df WHERE Sub_segment LIKE '%Offline_%' GROUP BY agreement_date""")
df_of

,Sub_segment,agreement_date,znam_fpd0,chis_fpd0,znam_fpd5,chis_fpd5,znam_fpd7,chis_fpd7
0,Offline_Direct,2026-01-01 00:00:00.000000,None,None,None,None,None,None
1,Offline_Direct,2026-08-30 00:00:00.000000,None,None,None,None,None,None
2,Offline_standart,2026-08-31 00:00:00.000000,None,None,None,None,None,None
3,Offline_Direct,2026-09-02 00:00:00.000000,None,None,None,None,None,None
4,Offline_standart,2026-09-03 00:00:00.000000,None,None,None,None,None,None


In [48]:
df['segment'] = df_of['Sub_segment'].apply(
    lambda x: 'Online' if 'Online' in x else 'Offline'
)
result = df.groupby(['segment', 'agreement_date']).agg(
    count=('chis_fpd0', 'count'),
    chis_fpd0_sum=('chis_fpd0', 'sum'),
    znam_fpd0_sum=('znam_fpd0', 'sum'),
).reset_index()
result

,segment,agreement_date,count,chis_fpd0_sum,znam_fpd0_sum
0,Offline,2026-01-01,0,0.0,0.0
1,Offline,2026-08-30,0,0.0,0.0
